# TP 1 - Du texte brut au premier modèle bigramme

Objectif : transformer un corpus en prédictions supervisées, puis entraîner une table qui prédit le caractère suivant.

## Étape 1 - Télécharger et inspecter le corpus

In [1]:
from urllib.request import urlretrieve
import os

url = ("https://raw.githubusercontent.com/karpathy/"
       "char-rnn/master/data/tinyshakespeare/input.txt")
urlretrieve(url, "input.txt")
print(os.path.abspath("input.txt"))

# lecture du fichier en entier
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(len(text))
print(text[:1000])

/content/input.txt
1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, n

In [2]:
# quelques chiffres sur le corpus
print("caractères      :", len(text))
print("retours ligne   :", text.count("\n"))
print("majuscules      :", sum(c.isupper() for c in text))
print("minuscules      :", sum(c.islower() for c in text))
# tout ce qui n'est ni lettre, ni chiffre, ni espace
print("ponctuation     :", sorted(c for c in set(text) if not c.isalnum() and not c.isspace()))

caractères      : 1115394
retours ligne   : 40000
majuscules      : 88754
minuscules      : 762324
ponctuation     : ['!', '$', '&', "'", ',', '-', '.', ':', ';', '?']


**Relevé :** 1 115 394 caractères, dont 88 754 majuscules et 762 324 minuscules. Il y a 40 000 retours à la ligne, et la ponctuation et les symboles sont `! $ & ' , - . : ; ?`. Le texte alterne un nom de personnage suivi de `:`, sa réplique, puis une ligne vide.

**Quelle est l'unité d'un token ?** Ici c'est le caractère : chaque lettre, espace, signe de ponctuation ou retour à la ligne est un token.

**Pourquoi `encoding="utf-8"` ?** Pour décoder les octets du fichier de la même façon sur toutes les machines. Sans ça, l'encodage par défaut dépend du système.

**Le fichier est-il déjà séparé en exemples ?** Non, c'est une seule longue chaîne de caractères. C'est nous qui découpons les exemples.

## Étape 2 - Passer des caractères aux entiers

In [3]:
# vocabulaire = tous les caractères différents, triés
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)
print("".join(chars))

# caractère -> entier, et l'inverse
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]                 # texte -> liste d'entiers
decode = lambda ids: ''.join([itos[i] for i in ids])    # liste d'entiers -> texte

message = "hii there"
print(encode(message))
print(decode(encode(message)))

# vérification : on retrouve bien le message de départ
assert decode(encode(message)) == message

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


**Pourquoi les tokens sont-ils des entiers ?** Le modèle ne manipule que des nombres. L'entier sert d'indice pour aller chercher une ligne dans une table.

**Pourquoi trier `chars` ?** L'ordre d'un `set` peut changer d'une exécution à l'autre. En triant, chaque caractère garde toujours le même identifiant.

**Que garantit `decode(encode(message)) == message` ?** Que les deux fonctions sont inverses sur ce message : aucune information n'est perdue. Un caractère absent du vocabulaire provoquerait une erreur.

## Étape 3 - Fabriquer les prédictions attendues

In [4]:
import torch

# tout le texte sous forme d'un tenseur d'entiers
data = torch.tensor(encode(text), dtype=torch.long)

# 90% pour l'entraînement, 10% pour la validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8
x = train_data[:block_size]            # entrée
y = train_data[1:block_size + 1]       # cible : décalée d'un caractère

for t in range(block_size):
    context = x[:t + 1]                # tout ce qu'on a vu jusqu'à t
    target = y[t]                      # le caractère à deviner
    print(context, "->", target)
    print("   ", repr(decode(context.tolist())), "->", repr(decode([target.item()])))

tensor([18]) -> tensor(47)
    'F' -> 'i'
tensor([18, 47]) -> tensor(56)
    'Fi' -> 'r'
tensor([18, 47, 56]) -> tensor(57)
    'Fir' -> 's'
tensor([18, 47, 56, 57]) -> tensor(58)
    'Firs' -> 't'
tensor([18, 47, 56, 57, 58]) -> tensor(1)
    'First' -> ' '
tensor([18, 47, 56, 57, 58,  1]) -> tensor(15)
    'First ' -> 'C'
tensor([18, 47, 56, 57, 58,  1, 15]) -> tensor(47)
    'First C' -> 'i'
tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> tensor(58)
    'First Ci' -> 't'




1. **Combien de prédictions produit ce bloc ?** 8, une par position (`block_size`).
2. **Pourquoi ne pas mélanger train et validation ?** La validation sert à mesurer le modèle sur du texte jamais vu. Si on mélange, le score est trop optimiste.
3. **Pour t = 3 ?** Le contexte est formé des 4 premiers caractères (`x[:4]`, soit `"Firs"`) et la cible est le 5e (`y[3]`, soit `"t"`).

## Étape 4 - Échantillonner un mini-batch

In [5]:
torch.manual_seed(1337)

batch_size = 4   # B : nombre de séquences en parallèle
block_size = 8   # T : longueur d'une séquence


def get_batch(split):
    source = train_data if split == "train" else val_data
    # positions de départ tirées au hasard
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([
        source[i : i + block_size] for i in starts
    ])
    # même chose, décalée d'un caractère
    y = torch.stack([
        source[i + 1 : i + block_size + 1] for i in starts
    ])
    return x, y


xb, yb = get_batch("train")
print(xb.shape, yb.shape)   # (4, 8) (4, 8)
print(xb)
print(yb)

torch.Size([4, 8]) torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


1. **Pourquoi retirer `block_size` à la borne de tirage ?** Il faut assez de caractères après le départ : `y` va jusqu'à la position `i + block_size`. La borne haute de `randint` est exclue, donc on ne dépasse jamais la fin.
2. **Avec B = 32 ?** Les tenseurs passent à (32, 8) : 256 prédictions par étape au lieu de 32. Le gradient est moins bruité, mais chaque étape coûte plus cher.
3. **Avec T = 64 ?** Les tenseurs passent à (4, 64) : un contexte plus long. Le modèle bigramme n'en profite pas car il ne regarde que le caractère courant, contrairement à un Transformer.

## Étapes 5 et 6 - Le modèle bigramme

In [6]:
import torch.nn as nn
from torch.nn import functional as F


class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # table (C, C) : une ligne de scores par caractère
        self.token_embedding_table = nn.Embedding(
            vocab_size, vocab_size
        )

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        # idx: (B,T)  ->  logits: (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)        # une ligne par prédiction
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    @torch.no_grad()                             # pas de gradient pour générer
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)                # (B, T, C)
            logits = logits[:, -1, :]            # dernier pas de temps : (B, C)
            probs = F.softmax(logits, dim=-1)    # scores -> probabilités
            idx_next = torch.multinomial(        # tirage au hasard : (B, 1)
                probs, num_samples=1
            )
            idx = torch.cat((idx, idx_next), dim=1)   # on l'ajoute à la suite
        return idx


torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size)

logits, loss = model(xb, yb)
print(logits.shape)     # (B*T, C)
print(loss.item())
print("nombre de paramètres :", sum(p.numel() for p in model.parameters()))

torch.Size([32, 65])
4.878634929656982
nombre de paramètres : 4225




**Pourquoi une table C × C ?** Il y a une ligne par caractère courant et, dans chaque ligne, un score pour chacun des C caractères suivants possibles. Ici 65 × 65 = 4 225 paramètres.

**Où retrouvera-t-on `d_emb` et `lm_head` au TP 4 ?** La table d'embedding sera de taille (C, d_emb) et donnera un vecteur de dimension d_emb. La couche `lm_head` (d_emb vers C) transformera ensuite ce vecteur en scores sur le vocabulaire.

## Génération avant entraînement

In [7]:
def generer_texte(n=300):
    depart = torch.zeros((1, 1), dtype=torch.long)        # id 0 = retour à la ligne
    with torch.random.fork_rng(devices=[]):               # ne perturbe pas les batchs
        torch.manual_seed(1)                              # même tirage à chaque fois
        sortie = model.generate(depart, max_new_tokens=n)
    return decode(sortie[0].tolist())


@torch.no_grad()
def estimer_loss(n_batchs=200):
    # moyenne sur plusieurs batchs : un seul batch est trop bruité
    res = {}
    for split in ["train", "val"]:
        pertes = [model(*get_batch(split))[1].item() for _ in range(n_batchs)]
        res[split] = sum(pertes) / len(pertes)
    return res


import math
print("loss si on répondait au hasard (uniforme) :", math.log(vocab_size))

pertes = {0: estimer_loss()}
textes = {0: generer_texte()}
print(pertes[0])
print(textes[0])

loss si on répondait au hasard (uniforme) : 4.174387269895637
{'train': 4.73025239944458, 'val': 4.72139356136322}

CCuw;suqW
r&!TqUuuOpx,O,SKuCjehkZyKA cUzPmsBkzKZXa;vLfqKSVSPiBCGvOW$N!r.imIoxZ;aXjaF$XgaM?OWwNXEs$:TpgwKtgRfZGZylo&BL:Dq tV3SZRb?a'daVhiToiSVTErdv$PsrJHRy!zUXlt-gRmNcCmO qbkErzLgCDM:CCXNLfNFfZ,w!BUQECn!mWAkOIl,v'Xd,ivafx?$ MtehsCE!
SK$h &l;wJYWdkpIQUF!a'ZehsJUAcNKl,sCIszgC'uiq,u3n$VXOcO dZMB
,puCKL&


## Étape 7 - Entraîner

In [8]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3
)


def entrainer(n_etapes):
    for step in range(n_etapes):
        xb, yb = get_batch("train")
        logits, loss = model(xb, yb)              # forward
        optimizer.zero_grad(set_to_none=True)     # on efface les anciens gradients
        loss.backward()                           # rétropropagation
        optimizer.step()                          # mise à jour des poids
    print("dernière loss :", loss.item())


# 100 étapes
entrainer(100)
pertes[100] = estimer_loss()
textes[100] = generer_texte()
print(pertes[100])
print(textes[100])

dernière loss : 4.9126715660095215
{'train': 4.6479066729545595, 'val': 4.653186559677124}

CCuw;suqW
r&!TqUuuOpx,O,SKuCjehkZyKA cUzPmsBkzKZXa;vLfqKSVSPiBCGvOW$N!r.imIoxZ;aXjaF$XgaM?OWwNXEs$:TpgwKtgRfZGZylo&BL:Dq tV3SZRb?a'daVhiToiSVvErdv$PsrJHRy!zUXlt-gRmNcCmO qbk qzLgCDM:CCXNLfNFfZ,w!BUQECn!mWAkOIl,v'Xd,ivafx?$ MtehsCE!
SK$h &l;wJYWdkpIQUF!a'ZehsJUAcNKl,sCIszgC'uiq,u3n$VXOcO dZMB
,puCKL&


In [9]:
# 900 étapes de plus, soit 1000 au total
entrainer(900)
pertes[1000] = estimer_loss()
textes[1000] = generer_texte()
print(pertes[1000])
print(textes[1000])

dernière loss : 4.058818817138672
{'train': 4.033849744796753, 'val': 4.036294572353363}

CCuw;suqW
r&!TqUuuOpx,O,SKuCjehoZyKA cUzPmsowM:PGDWvLf ERTSPiBCHxJW$N!r.imIoxZ;aXjaF$XgagFNWwNXEs$:TpgwKtgRfZGLToo&BL:Dq tV3SZRb?a'daVhiToiSVvErdv$PsrJHRy!zUXlt-gRmNcCmO qbk izLgi'M:CCXNLf -fZ,w!BUQECnnmmOnkyl,v'Xd,ivaf.?$ MtereCE!
FD$h j
pwJYWdkph
SF!a'Zeh pUAcNKl,sCIs,lC'uiq,u3n$VXOcO dZMB
,puCKL&


## Comparaison

In [10]:
# récapitulatif
for etapes in [0, 100, 1000]:
    p = pertes[etapes]
    print(f"{etapes:5d} étapes | loss train = {p['train']:.3f} | loss val = {p['val']:.3f}")

for etapes in [0, 100, 1000]:
    print(f"\n===== après {etapes} étapes =====")
    print(textes[etapes])

    0 étapes | loss train = 4.730 | loss val = 4.721
  100 étapes | loss train = 4.648 | loss val = 4.653
 1000 étapes | loss train = 4.034 | loss val = 4.036

===== après 0 étapes =====

CCuw;suqW
r&!TqUuuOpx,O,SKuCjehkZyKA cUzPmsBkzKZXa;vLfqKSVSPiBCGvOW$N!r.imIoxZ;aXjaF$XgaM?OWwNXEs$:TpgwKtgRfZGZylo&BL:Dq tV3SZRb?a'daVhiToiSVTErdv$PsrJHRy!zUXlt-gRmNcCmO qbkErzLgCDM:CCXNLfNFfZ,w!BUQECn!mWAkOIl,v'Xd,ivafx?$ MtehsCE!
SK$h &l;wJYWdkpIQUF!a'ZehsJUAcNKl,sCIszgC'uiq,u3n$VXOcO dZMB
,puCKL&

===== après 100 étapes =====

CCuw;suqW
r&!TqUuuOpx,O,SKuCjehkZyKA cUzPmsBkzKZXa;vLfqKSVSPiBCGvOW$N!r.imIoxZ;aXjaF$XgaM?OWwNXEs$:TpgwKtgRfZGZylo&BL:Dq tV3SZRb?a'daVhiToiSVvErdv$PsrJHRy!zUXlt-gRmNcCmO qbk qzLgCDM:CCXNLfNFfZ,w!BUQECn!mWAkOIl,v'Xd,ivafx?$ MtehsCE!
SK$h &l;wJYWdkpIQUF!a'ZehsJUAcNKl,sCIszgC'uiq,u3n$VXOcO dZMB
,puCKL&

===== après 1000 étapes =====

CCuw;suqW
r&!TqUuuOpx,O,SKuCjehoZyKA cUzPmsowM:PGDWvLf ERTSPiBCHxJW$N!r.imIoxZ;aXjaF$XgagFNWwNXEs$:TpgwKtgRfZGLToo&BL:Dq tV3SZRb?a'daVhiToiSVvErdv$

## Bilan

**Pourquoi ne garder que le dernier pas de temps à la génération ?** On veut le caractère qui suit la séquence. Seuls les scores de la dernière position le prédisent ; les autres positions prédisent des caractères déjà connus.

**Pourquoi aplatir B × T pour la loss ?** Chaque position est une classification indépendante. `cross_entropy` attend des logits (N, C) et des cibles (N), avec ici N = B × T, et renvoie la moyenne.



**Comparaison avant / après**

| Étapes | Loss train | Loss validation |
|---:|---:|---:|
| 0 | 4,730 | 4,721 |
| 100 | 4,648 | 4,653 |
| 1 000 | 4,034 | 4,036 |

- **Au départ,** la loss (4,73) dépasse celle d'un choix uniforme, ln(65) = 4,17 : les poids aléatoires donnent de fortes préférences au mauvais endroit.
- **Après 100 étapes,** la baisse est faible (environ 0,07 à 0,08) et le modèle reste moins bon que le choix uniforme.
- **Après 1 000 étapes,** la loss a baissé d'environ 0,7 et passe sous 4,17 : la table a appris quelque chose sur les enchaînements de caractères.
- **Train et validation** restent presque identiques (écart inférieur à 0,01) : pas de sur-apprentissage visible.
- La « dernière loss » affichée après 100 étapes (4,91) ne montre pas une dégradation : elle porte sur un seul batch de 32 prédictions, très bruité. On compare donc les moyennes sur 200 batchs.

**Structure du texte**

- **Avant :** suite de caractères au hasard, majuscules au milieu des mots, nombreux symboles, aucun mot.
- **Après 100 étapes :** presque le même texte (3 caractères différents sur 301).
- **Après 1 000 étapes :** environ 50 positions changent. On voit un peu plus d'espaces (de 6 à 10) et de retours à la ligne (de 4 à 6), mais toujours pas d'anglais lisible ni de structure « nom : réplique ».
- **Pourquoi les textes se ressemblent :** la génération utilise la même graine à chaque fois. Les tirages sont identiques, donc seul l'effet des poids modifiés apparaît.

Avec `lr = 1e-3`, 1 000 étapes ne suffisent pas pour obtenir un texte lisible, même si la loss progresse nettement.

**Limite du modèle :** il ne regarde qu'un seul caractère pour prédire le suivant. Même entraîné longtemps, il ne peut pas tenir compte du contexte : c'est le rôle de l'attention (TP 2).